# 🌍 Member 1 — Extract: REST Countries API
**Business Goal:** Extract country data to understand global market demographics.

**Pipeline Role:** SOURCE 1 → saves `countries_raw.parquet`

In [1]:
import requests
import pandas as pd
import os
import json
from datetime import datetime

print('✅ Libraries loaded')
print(f'📅 Run time: {datetime.now()}')

✅ Libraries loaded
📅 Run time: 2026-05-12 12:15:31.569963


In [2]:
# ── Create output folders ──────────────────────────────────────────
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print('📁 Folders ready')

📁 Folders ready


In [3]:
# ── EXTRACT: Call REST Countries API ──────────────────────────────
URL = 'https://restcountries.com/v3.1/all?fields=name,population,area,region,subregion,currencies,capital'

print('🌐 Calling REST Countries API...')
response = requests.get(URL, timeout=30)

if response.status_code == 200:
    countries_raw = response.json()
    print(f'✅ Success! Retrieved {len(countries_raw)} countries')
else:
    raise Exception(f'❌ API Error: {response.status_code}')

🌐 Calling REST Countries API...
✅ Success! Retrieved 250 countries


In [4]:
# ── PARSE: Flatten nested JSON into a clean DataFrame ─────────────
records = []

for c in countries_raw:
    # Extract currency code and name safely
    currencies = c.get('currencies', {})
    currency_code = list(currencies.keys())[0] if currencies else 'N/A'
    currency_name = currencies[currency_code]['name'] if currencies and currency_code != 'N/A' else 'N/A'

    records.append({
        'country_name':   c['name']['common'],
        'official_name':  c['name']['official'],
        'capital':        c['capital'][0] if c.get('capital') else 'N/A',
        'region':         c.get('region', 'N/A'),
        'subregion':      c.get('subregion', 'N/A'),
        'population':     c.get('population', 0),
        'area_km2':       c.get('area', 0.0),
        'currency_code':  currency_code,
        'currency_name':  currency_name,
        'extracted_at':   datetime.now().isoformat()
    })

df_countries = pd.DataFrame(records)
print(f'✅ Parsed {len(df_countries)} countries into DataFrame')
print(f'📊 Columns: {list(df_countries.columns)}')

✅ Parsed 250 countries into DataFrame
📊 Columns: ['country_name', 'official_name', 'capital', 'region', 'subregion', 'population', 'area_km2', 'currency_code', 'currency_name', 'extracted_at']


In [5]:
# ── PREVIEW: See what we extracted ────────────────────────────────
print('\n📋 SAMPLE DATA:')
print(df_countries.head(10).to_string())

print('\n📊 DATA SUMMARY:')
print(f'  Total countries:  {len(df_countries)}')
print(f'  Regions:          {df_countries["region"].nunique()}')
print(f'  Unique currencies:{df_countries["currency_code"].nunique()}')
print(f'  Countries with no area: {(df_countries["area_km2"] == 0).sum()}')

print('\n🌍 Countries per Region:')
print(df_countries['region'].value_counts().to_string())


📋 SAMPLE DATA:
       country_name              official_name         capital    region         subregion  population   area_km2 currency_code             currency_name                extracted_at
0          Anguilla                   Anguilla      The Valley  Americas         Caribbean       16010       91.0           XCD  Eastern Caribbean dollar  2026-05-12T12:16:01.053467
1         Guatemala      Republic of Guatemala  Guatemala City  Americas   Central America    18079810   108889.0           GTQ        Guatemalan quetzal  2026-05-12T12:16:01.053467
2            Gambia     Republic of the Gambia          Banjul    Africa    Western Africa     2422712    10689.0           GMD                    dalasi  2026-05-12T12:16:01.053467
3            Mexico      United Mexican States     Mexico City  Americas     North America   130575786  1964375.0           MXN              Mexican peso  2026-05-12T12:16:01.053467
4            Malawi         Republic of Malawi        Lilongwe    Africa  

In [6]:
# ── QUALITY CHECK ─────────────────────────────────────────────────
print('🔍 DATA QUALITY CHECKS:')
print(f'  Null values:\n{df_countries.isnull().sum()}')
print(f'\n  Dtypes:\n{df_countries.dtypes}')

# Fix: remove rows where population is 0 (uninhabited territories)
before = len(df_countries)
df_countries = df_countries[df_countries['population'] > 0]
after = len(df_countries)
print(f'\n  Removed {before - after} uninhabited territories')
print(f'  Final record count: {after}')

🔍 DATA QUALITY CHECKS:
  Null values:
country_name     0
official_name    0
capital          0
region           0
subregion        0
population       0
area_km2         0
currency_code    0
currency_name    0
extracted_at     0
dtype: int64

  Dtypes:
country_name      object
official_name     object
capital           object
region            object
subregion         object
population         int64
area_km2         float64
currency_code     object
currency_name     object
extracted_at      object
dtype: object

  Removed 5 uninhabited territories
  Final record count: 245


In [7]:
# ── SAVE: Write raw extracted data to Parquet ─────────────────────
OUTPUT_PATH = 'data/raw/countries_raw.parquet'
df_countries.to_parquet(OUTPUT_PATH, index=False)

# Verify saved file
df_verify = pd.read_parquet(OUTPUT_PATH)
print(f'✅ Saved to: {OUTPUT_PATH}')
print(f'✅ Verified: {len(df_verify)} rows, {len(df_verify.columns)} columns')
print(f'💾 File size: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB')
print('\n🏁 Member 1 COMPLETE — countries_raw.parquet is ready for Member 3 (PySpark Transform)')

✅ Saved to: data/raw/countries_raw.parquet
✅ Verified: 245 rows, 10 columns
💾 File size: 23.3 KB

🏁 Member 1 COMPLETE — countries_raw.parquet is ready for Member 3 (PySpark Transform)
